# Convolutional Neural Networks (CNNs)
Building a CNN in PyTorch and training on MNIST handwritten digits.

## 1. Theory & Intuition

### Why CNNs over Regular Neural Networks?

**Problem with regular NNs for images:**
- A 1000×1000 image with 512 neurons = 500 million weights — untrainable
- Flattening destroys spatial information

**What CNNs do differently:**
- Small filter (3×3) slides across image — same weights applied everywhere (weight sharing)
- Each filter learns one pattern: edges, curves, color gradients
- Stack layers → detect complex patterns

**Output size formula:** `(input - filter + 2*padding) / stride + 1`

**Architecture:**
Input (1×28×28) → Conv1(8 filters) → ReLU → MaxPool → 8×14×14 → Conv2(16 filters) → ReLU → MaxPool → 16×7×7 → Flatten(784) → Linear → 10 classes

## 2. From Scratch Implementation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.fc = nn.Linear(16 * 7 * 7, 10)
    
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

model = SimpleCNN()
print(model)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Load Data — MNIST

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")
print(f"Batches per epoch: {len(train_loader)}")

## 4. Train & Evaluate

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(3):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {running_loss/len(train_loader):.4f}")

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        output = model(images)
        _, predicted = torch.max(output, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

## 5. Key Takeaways

### Results
| Metric | Value |
|--------|-------|
| Dataset | MNIST (60k train, 10k test) |
| Epochs | 3 |
| Final Loss | ~0.07 |
| Test Accuracy | ~98% |

### Regular NN vs CNN
| | Regular NN | CNN |
|--|-----------|-----|
| Parameters | Millions | Much fewer (weight sharing) |
| Spatial info | Lost | Preserved |
| MNIST accuracy | ~97% | ~98%+ |

### Key Points
1. Weight sharing — same filter applied everywhere, fewer parameters
2. Output size = (input - filter + 2*padding) / stride + 1
3. MaxPool halves spatial dimensions
4. x.view(batch_size, -1) flattens before Linear layer
5. CrossEntropyLoss for multiclass, BCELoss for binary
6. torch.max(output, 1) picks class with highest score
7. Always use model.eval() + torch.no_grad() during evaluation